# From Local Tools to Managed APIs, Hands On

Earlier labs gave the agent its tools with `@tool` decorators, so the tool code lived *inside* the agent.
This notebook takes them out. Horizon Travel's booking data sits behind a Lambda function, and the agent
reaches it through an **AgentCore Gateway** that speaks MCP and discovers the tools automatically — with no
`@tool` code in the agent at all.

It rebuilds `demo.py` one idea at a time, so each piece can be run and watched rather than read. It is a
lesson that happens to arrive at the same code, not a copy of it.

**Part A — no Gateway, no AWS resources.** Steps 1 to 5 run in this kernel. They make no service calls at
all until the very end of Part A, where a Bedrock model call is clearly marked. They create nothing.

**Part B — the real Gateway.** Deploying the Lambda and creating the Gateway are real AWS resources, and
that part is gated: we stop and decide before building anything.

Run the cells in order. Each step ends with output you can check against what the markdown predicted.

## Step 1 — Setup, and where credentials come from

This notebook eventually calls Amazon Bedrock (for the model) and AgentCore Gateway (for the tools). Both
need AWS credentials, and **the notebook never contains them**. boto3 resolves them from the environment,
and on this machine that environment is redirected to `~/.aws-personal/` by two variables.

`demo.py` runs deployed, where AgentCore Runtime supplies credentials. Here in a kernel we point boto3 at the
personal credential store ourselves — as the first thing, before anything constructs a client, because a
boto3 session caches whatever it resolves on first use.

Everything printed by this cell is designed to be safe to commit: notebook outputs are saved inside the
`.ipynb`, so paths are shown relative to the repo root and identifiers are masked. The masking helpers are
defined here and used throughout Part B.

**What to expect:** two package versions, a repo-relative working directory, and a masked identity/account.
The final four account digits should be `6087`, identifying the Vocareum lab account that pays for later
model calls, without publishing the full identifier in notebook output.
`NO CREDENTIALS FOUND` means the kernel was launched without the environment; relaunch it from a shell that
has the two paths set.

Free — the only call here is a read-only `get_caller_identity`.

In [1]:
# =====================================================
# 1. IMPORTS AND CREDENTIAL RESOLUTION
# =====================================================
import json
import logging
import os
from importlib.metadata import version
from pathlib import Path

# Point boto3 at the personal credential store BEFORE any client is built. A boto3
# session caches what it resolves on first use, so this has to come first.
os.environ["AWS_CONFIG_FILE"] = os.path.expanduser("~/.aws-personal/config")
os.environ["AWS_SHARED_CREDENTIALS_FILE"] = os.path.expanduser("~/.aws-personal/credentials")

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

REGION = "us-east-1"


# =====================================================
# 2. LOGGING - for a kernel, not a container
# =====================================================
logging.basicConfig(
    level=logging.WARNING,               # keeps botocore and urllib3 quiet
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,                          # override any handler the kernel installed
)
logger = logging.getLogger("WanderBot.Gateway")
logger.setLevel(logging.INFO)


# =====================================================
# 3. PRINTING HELPERS - keep saved output publishable
# =====================================================
# Notebook outputs are committed inside the .ipynb, so two things need care:
# absolute paths (they carry the machine's username) and AWS identifiers.


def repo_relative(path):
    """Render a path relative to the repository root.

    Walks up to the directory containing .git, so the home prefix stays out of
    saved output while the useful part of the path remains.

    Args:
        path: A filesystem path, str or Path.

    Returns:
        str: The path relative to the repo root, or just the leaf if it is not
            inside a repository.
    """
    path = Path(path)
    for parent in [path, *path.parents]:
        if (parent / ".git").is_dir():
            return str(path.relative_to(parent))
    return path.name


def mask_id(value, keep_last=4):
    """Mask an identifier, keeping the final characters.

    Enough tail is left to recognise which account or resource this is when
    comparing against the console, not enough to reuse.

    Args:
        value: The identifier to mask.
        keep_last: How many trailing characters to leave visible.

    Returns:
        str: The value with all but its last keep_last characters replaced by
            asterisks.
    """
    s = str(value)
    if len(s) <= keep_last:
        return "*" * len(s)
    return "*" * (len(s) - keep_last) + s[-keep_last:]


def mask_arn(arn):
    """Mask an ARN's account id and unique suffix, keeping its structure.

    The service, region, resource type and human-chosen name stay readable; the
    account id and any generated suffix are hidden.

    Args:
        arn: An ARN string.

    Returns:
        str: The ARN with account and suffix masked, or mask_id(arn) when the
            input is not ARN-shaped.
    """
    parts = str(arn).split(":")
    if len(parts) < 6 or parts[0] != "arn":
        return mask_id(arn)
    parts[4] = mask_id(parts[4])                        # account id
    kind, sep, name = parts[5].partition("/")
    if sep:
        head, dash, tail = name.rpartition("-")
        parts[5] = f"{kind}/{head}{dash}{'*' * len(tail)}" if dash else f"{kind}/{mask_id(name)}"
    return ":".join(parts)


# =====================================================
# 4. PATHS - this notebook's own directory
# =====================================================
BASE_DIR = Path.cwd()
# The Lambda source and schema sit beside this notebook; fall back one level up
# in case the kernel started in the repo root.
CANDIDATES = [BASE_DIR, BASE_DIR / "lab_5" / "Demo"]
DEMO_DIR = next((p for p in CANDIDATES if (p / "booking_lambda.py").is_file()), None)


# =====================================================
# 5. VERIFY - every assumption the rest of the notebook rests on
# =====================================================
print("strands-agents   :", version("strands-agents"))
print("mcp              :", version("mcp"))
print("demo dir         :", repo_relative(DEMO_DIR) if DEMO_DIR else "NOT FOUND")

if DEMO_DIR is None:
    raise FileNotFoundError("booking_lambda.py not found beside this notebook")

try:
    who = boto3.client("sts", region_name=REGION).get_caller_identity()
    print("identity         :", mask_arn(who["Arn"]))
    print("account          :", mask_id(who["Account"]))
    print("\nready for Step 2.")
except NoCredentialsError:
    print("\nNO CREDENTIALS FOUND - boto3 located no usable credential file.")
    print("The cause is this kernel's environment, not the code, and no cell can repair it:")
    print("  1. quit the IDE completely (Cmd-Q, not just closing the window)")
    print("  2. relaunch it from a shell where AWS_CONFIG_FILE / AWS_SHARED_CREDENTIALS_FILE are set")
    print("  3. reopen this notebook and Restart Kernel, then re-run from the top")
except ClientError as exc:
    print("\nCredentials were FOUND but AWS rejected them:", exc.response["Error"]["Code"])
    print("Resolution worked; the credentials themselves need refreshing.")

strands-agents   : 1.53.0
mcp              : 1.29.1
demo dir         : lab_5/Demo
identity         : arn:aws:sts::********6087:assumed-role/***************************8564
account          : ********6087

ready for Step 2.


## Step 2 — Why take the tools out of the agent?

In the earlier labs a tool was a decorated function living in the agent's own source: `@tool` above a
Python function, and Strands turned its signature and docstring into a schema the model could call. That is
the right thing for development — one file, nothing to deploy, easy to change.

It is the wrong thing for production, for reasons that have nothing to do with the model:

- **The tool and the agent share a lifecycle.** A fix to the booking logic means redeploying the agent, and
  every agent that wants that tool carries its own copy of the code.
- **The tool inherits the agent's runtime.** It runs with the agent's Python version, its dependencies, its
  permissions — even when the tool is really a call into a separate system with separate needs.
- **The data lives elsewhere anyway.** Horizon's bookings are not in the agent. They are behind a service,
  and a `@tool` that reaches them is just a client with the agent's deployment schedule bolted on.

The production shape is to run the tool as its own service — here a Lambda function — and put something
between the agent and that service that both sides understand. That something is the **AgentCore Gateway**.
It speaks MCP (Model Context Protocol) to the agent, invokes the Lambda on the other side, and — the part
that does the work — **discovers the available tools and describes them to the agent automatically**, so the
agent carries no tool code at all.

```
WanderBot (Strands agent)
    |  MCP  (tool discovery + tool calls)
AgentCore Gateway
    |  direct Lambda invocation
booking_lambda.py   (get_booking, list_bookings_by_email)
```

The rest of Part A takes this apart from the bottom up, without a Gateway: first the Lambda as a plain
function (Step 3), then the schema that tells the Gateway what the Lambda offers (Step 4), then the MCP
client the agent uses to discover it (Step 5). Part B wires them together for real.

The cell below is a one-off recap of the *old* way, so the contrast is on the page rather than in memory.

In [2]:
# =====================================================
# RECAP - what a tool looked like when it lived in the agent
# =====================================================
# This is the pattern from the earlier labs. It is here only for contrast; nothing
# in lab 5 uses it. Note where the logic lives: right here, in the agent's process.
from strands import tool


@tool
def get_booking_local(booking_ref: str) -> str:
    """Retrieve a booking by reference (illustrative local version).

    Args:
        booking_ref: Booking reference such as BK-1001.

    Returns:
        A short string. Real logic omitted - the point is only where it would run.
    """
    return f"(pretend booking details for {booking_ref})"


# The decorator wraps the function into a Strands tool. The schema the model sees
# is built from the signature and docstring above - all of it inside the agent.
print("tool name        :", get_booking_local.tool_name)
print("schema keys      :", sorted(get_booking_local.tool_spec.keys()))
print("input properties :", list(get_booking_local.tool_spec["inputSchema"]["json"]["properties"]))
print()
print("The name, the description and the parameter schema all come from code in")
print("this process. Step 3 onward moves every bit of that out to a Lambda.")

tool name        : get_booking_local
schema keys      : ['description', 'inputSchema', 'name']
input properties : ['booking_ref']

The name, the description and the parameter schema all come from code in
this process. Step 3 onward moves every bit of that out to a Lambda.


## Step 3 — The Lambda is just a function

Before any Gateway exists, look at what sits behind it. `booking_lambda.py` is an ordinary Python module
with two functions over some hard-coded data — three sample bookings — and a handler. There is no HTTP
routing, no web framework, nothing AWS-specific in the functions themselves. That is the point the whole
lesson turns on: the tool is a plain function, and everything else is transport.

This step imports that real file and runs it *in this kernel*, with no AWS involved at all. If the logic is
sound here, then anything that goes wrong in Part B is the Gateway or the deployment — not the tool.

### 3a. Call the functions directly

`get_booking` looks up one booking by reference; `list_bookings_by_email` finds all bookings for an address.
Both return JSON strings. The cell imports the module from the demo directory and calls them the way any
Python code would.

**What to expect:** Alice's BK-1001 returned as JSON, both of Alice's bookings for her email, and a plain
not-found message for a reference that does not exist. No credentials are used.

Free — pure local calls.

In [3]:
# =====================================================
# 3a. THE LAMBDA'S FUNCTIONS, CALLED IN-PROCESS
# =====================================================
import sys

# The Lambda file sits in the demo directory; make it importable.
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

import booking_lambda

# A single lookup by reference.
print("get_booking('BK-1001'):")
print("  ", booking_lambda.get_booking("BK-1001"))

# The handler upper-cases the reference, so lookup is case-insensitive.
print()
print("get_booking('bk-1001')  (lower case still resolves):")
print("  ", booking_lambda.get_booking("bk-1001")[:60], "...")

# A reference that does not exist - a message, not an exception.
print()
print("get_booking('BK-9999'):")
print("  ", booking_lambda.get_booking("BK-9999"))

# All bookings for one customer email.
print()
print("list_bookings_by_email('alice@example.com'):")
print("  ", booking_lambda.list_bookings_by_email("alice@example.com")[:90], "...")

print()
print("Plain functions over local data. Nothing here knows about a Gateway.")

get_booking('BK-1001'):
   {"booking_ref": "BK-1001", "customer": "Alice Johnson", "email": "alice@example.com", "flight": "HZ-101", "route": "LHR \u2192 CDG", "date": "2026-03-15", "cabin": "Economy Flex", "fare_usd": 189.99, "status": "CONFIRMED"}

get_booking('bk-1001')  (lower case still resolves):
   {"booking_ref": "BK-1001", "customer": "Alice Johnson", "ema ...

get_booking('BK-9999'):
   No booking found with reference BK-9999.

list_bookings_by_email('alice@example.com'):
   {"email": "alice@example.com", "bookings": [{"booking_ref": "BK-1001", "customer": "Alice  ...

Plain functions over local data. Nothing here knows about a Gateway.


### 3b. How the handler knows which function to call

The Gateway does not call `get_booking` directly. It invokes the *Lambda*, once, and tells the handler which
tool was meant. The handler reads that from the invocation's client context:

``` Python
raw_tool = context.client_context.custom.get("bedrockAgentCoreToolName", "")
tool = raw_tool.split("___", 1)[-1] if "___" in raw_tool else raw_tool
```

The Gateway passes a **prefixed** name like `booking-target___get_booking` — target name, three underscores,
tool name — and the handler keeps the part after the separator. That is the routing: one Lambda, several
tools, dispatched by a name the Gateway sets.

To exercise this without a Gateway, the cell builds a small fake `context` object with just the attributes
the handler reads, then calls the handler as the Gateway would. It also shows the fallback path — when there
is no client context at all, the handler infers the tool from the event's keys, which is what makes a plain
`aws lambda invoke` still work.

**What to expect:** the prefixed name routes to `get_booking` and returns a `200` with the booking in the
body; the fallback routes `{"email": ...}` to `list_bookings_by_email`. The handler prints its own
`Tool called: ...` line each time.

Free — still no AWS.

In [4]:
# =====================================================
# 3b. THE HANDLER'S TOOL-NAME ROUTING, VIA A FAKE CONTEXT
# =====================================================


class FakeClientContext:
    """The minimal client_context the handler reads from.

    A real Lambda context carries far more; the handler only touches
    client_context.custom.get("bedrockAgentCoreToolName"), so that is all this
    reproduces.
    """

    def __init__(self, tool_name):
        """Store the tool name under the key the Gateway uses.

        Args:
            tool_name: The value the Gateway would place in
                bedrockAgentCoreToolName, e.g. "booking-target___get_booking".
        """
        # .custom must be an object with a .get method; a dict provides exactly that.
        self.custom = {"bedrockAgentCoreToolName": tool_name}


class FakeContext:
    """Stand-in for the Lambda context object, carrying only client_context."""

    def __init__(self, tool_name):
        """Wrap a FakeClientContext.

        Args:
            tool_name: Passed through to FakeClientContext.
        """
        self.client_context = FakeClientContext(tool_name)


# ---- the Gateway path: a prefixed tool name in the client context ----
gateway_context = FakeContext("booking-target___get_booking")
result = booking_lambda.lambda_handler({"booking_ref": "BK-1001"}, gateway_context)

print("invoked as the Gateway would, tool = 'booking-target___get_booking'")
print("  statusCode:", result["statusCode"])
print("  body      :", result["body"][:80], "...")

# ---- the fallback path: no client context, tool inferred from event keys ----
print()
fallback = booking_lambda.lambda_handler({"email": "bob@example.com"}, None)
print("invoked with context=None (e.g. plain 'aws lambda invoke')")
print("  the handler saw an 'email' key and routed to list_bookings_by_email")
print("  body      :", fallback["body"][:80], "...")

Tool called: get_booking | Event: {"booking_ref": "BK-1001"}
invoked as the Gateway would, tool = 'booking-target___get_booking'
  statusCode: 200
  body      : {"result": "{\"booking_ref\": \"BK-1001\", \"customer\": \"Alice Johnson\", \"em ...

Tool called: list_bookings_by_email | Event: {"email": "bob@example.com"}
invoked with context=None (e.g. plain 'aws lambda invoke')
  the handler saw an 'email' key and routed to list_bookings_by_email
  body      : {"result": "{\"email\": \"bob@example.com\", \"bookings\": [{\"booking_ref\": \" ...


## Step 4 — The schema is the bridge

Step 3 showed the Lambda is just functions. But a function's name, its description and its parameters are
Python — the Gateway cannot see them, and neither can the model. Something has to *describe* the tools in a
form both understand. That something is the schema in `booking_lambda.json`.

You register it when you add the Lambda as a target to the Gateway. It is a list of tool definitions, one
per function the Lambda exposes, and each definition has three parts:

- **`name`** — the tool the model calls, and the suffix the Gateway puts after `___` in the client context
  (Step 3b). `get_booking` here is why the handler routed on `get_booking` there.
- **`description`** — prose the model reads when deciding whether this tool fits the request.
- **`inputSchema`** — a JSON Schema for the parameters, so the model knows what to send and the Gateway can
  validate it.

Here is the thing worth noticing. This is the *same* information Step 2's `@tool` produced — name,
description, parameter schema — and the *same* three-part shape. The decorator generated it from the
function's signature and docstring automatically. For a Gateway tool you write it by hand instead, because
the Gateway sits in a different process from the function and cannot introspect it. Same contract, authored
in a different place.

The cell reads the real schema file and lines its shape up against the tool spec from Step 2.

Free — reading a local file.

In [5]:
# =====================================================
# 4. THE TOOL SCHEMA THE GATEWAY REGISTERS
# =====================================================

schema_path = DEMO_DIR / "booking_lambda.json"
schema = json.loads(schema_path.read_text())

print("tools defined in", repo_relative(schema_path), ":", len(schema))
print()

for tool_def in schema:
    required = tool_def["inputSchema"].get("required", [])
    params = list(tool_def["inputSchema"]["properties"])
    print("  name        :", tool_def["name"])
    print("  description :", tool_def["description"][:64], "...")
    print("  parameters  :", params, " required:", required)
    print()

# The same three keys the decorator produced in Step 2, authored by hand here.
gateway_keys = sorted(schema[0].keys())
local_keys = sorted(get_booking_local.tool_spec.keys())
print("Gateway schema keys per tool :", gateway_keys)
print("Step 2 @tool tool_spec keys  :", local_keys)
print("same three-part contract     :", gateway_keys == local_keys)

tools defined in lab_5/Demo/booking_lambda.json : 2

  name        : get_booking
  description : Retrieve full details for a single booking using its booking ref ...
  parameters  : ['booking_ref']  required: ['booking_ref']

  name        : list_bookings_by_email
  description : List all bookings associated with a customer email address. ...
  parameters  : ['email']  required: ['email']

Gateway schema keys per tool : ['description', 'inputSchema', 'name']
Step 2 @tool tool_spec keys  : ['description', 'inputSchema', 'name']
same three-part contract     : True


## Step 5 — What does the agent actually connect to?

The architecture now has both ends. Step 3 proved the Lambda logic and routing; Step 4 supplied the contract
that describes its tools. What remains is the agent-facing half: open an MCP session to the Gateway, ask the
server which tools exist, and hand those tools to Strands.

Three layers are easy to blur together:

| Layer | Object in the demo | Responsibility |
| --- | --- | --- |
| **Transport** | `streamable_http_client(...)` | moves MCP messages over HTTP to the Gateway endpoint |
| **MCP session** | `MCPClient(...)` | owns connection lifetime, performs the MCP handshake, lists tools, calls tools |
| **Agent** | `Agent(tools=tools)` | gives schemas to the model and invokes whichever tool the model selects |

The Gateway is the **MCP server**. `MCPClient` is exactly what its name says: the client. Lambda never speaks
MCP itself — the Gateway translates the MCP tool call into a direct Lambda invocation.

Everything in this step is local. Without a Gateway endpoint we cannot prove what tools the server returns,
but we can prove when networking starts, what fails before a session exists, and why discovered tools fit
straight into `Agent(tools=...)`.

### 5a. Construct the client without connecting

The demo passes a function into `MCPClient`, not an open connection:

``` Python
MCPClient(lambda: streamable_http_client(url=GATEWAY_ENDPOINT))
```

That `lambda` is a **transport factory** — a recipe the client can call later. Construction stores the recipe;
it does not execute it. Entering `with client:` is the boundary where real work starts: `__enter__` calls
`start()`, starts a background thread, opens the transport, performs MCP initialisation, and blocks until the
session is ready or the startup timeout expires.

The cell makes that visible with a named factory and a counter. The endpoint uses `.invalid`, a reserved domain,
but no request should reach it because the context is never entered. It then calls `list_tools_sync()` too
early on purpose; the method should refuse locally with `MCPClientInitializationError` rather than attempting
network access.

**What to expect:** factory call count remains zero before and after the failed listing, and the error says the
client session is not running.

Free — no network call.

In [6]:
# =====================================================
# 5a. CONSTRUCTION IS LAZY; THE WITH BLOCK OPENS THE SESSION
# =====================================================
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp.mcp_client import MCPClient, MCPClientInitializationError

PLACEHOLDER_ENDPOINT = "https://example.invalid/mcp"
factory_calls = 0


def make_transport():
    """Build the streamable-HTTP transport when MCPClient asks for it.

    This function is the expanded, readable form of the demo's lambda. Merely
    passing it to MCPClient does not run it; entering `with client:` does.

    Returns:
        An asynchronous context manager that opens MCP streams to the placeholder
        endpoint when entered.
    """
    global factory_calls
    factory_calls = factory_calls + 1
    return streamable_http_client(url=PLACEHOLDER_ENDPOINT)


client = MCPClient(make_transport)

print("transport factory calls after MCPClient(...):", factory_calls)
print("supports 'with client:'                    :", hasattr(client, "__enter__"))
print("supports list_tools_sync()                 :", hasattr(client, "list_tools_sync"))

print()
print("calling list_tools_sync() before entering the context:")
try:
    client.list_tools_sync()
    print("  succeeded - unexpected")
except MCPClientInitializationError as error:
    print(" ", type(error).__name__ + ":", error)

print()
print("transport factory calls after that failure :", factory_calls)
print("Zero means the failure happened locally; no request reached example.invalid.")

transport factory calls after MCPClient(...): 0
supports 'with client:'                    : True
supports list_tools_sync()                 : True

calling list_tools_sync() before entering the context:
  MCPClientInitializationError: the client session is not running. Ensure the agent is used within the MCP client context manager. For more information see: https://strandsagents.com/docs/user-guide/concepts/tools/mcp-tools/#mcpclientinitializationerror-python

transport factory calls after that failure : 0
Zero means the failure happened locally; no request reached example.invalid.


### 5b. Why can discovered tools go straight into `Agent`?

`list_tools_sync()` receives MCP tool definitions from the server and wraps each one as an `MCPAgentTool`.
That class and Step 2's decorated local tool look unrelated — one calls a remote MCP server, the other calls a
Python function — but both implement Strands' `AgentTool` interface.

That common interface is the whole reason the agent code barely changes. `Agent` asks every tool for a
`tool_name`, a `tool_spec`, and a way to invoke it; it does not care which subclass supplies them. The model
sees the same name, description and input schema either way and cannot infer where execution happens.

The cell prints the inheritance chains for both types and checks the shared base class.

Free — type inspection only.

In [ ]:
# =====================================================
# 5b. LOCAL AND REMOTE TOOLS CONVERGE ON AgentTool
# =====================================================
import inspect

from strands.tools.mcp.mcp_agent_tool import MCPAgentTool
from strands.types.tools import AgentTool


def inheritance_chain(value_or_class):
    """Return readable class names from a value's inheritance hierarchy.

    Args:
        value_or_class: Either a class or an instance whose class should be
            inspected.

    Returns:
        list[str]: Class names from most specific to most general.
    """
    if inspect.isclass(value_or_class):
        class_to_inspect = value_or_class
    else:
        class_to_inspect = type(value_or_class)

    names = []
    for base in inspect.getmro(class_to_inspect):
        names.append(base.__name__)
    return names


print("Step 2 local tool:")
print("  chain:", " -> ".join(inheritance_chain(get_booking_local)))
print("  is AgentTool:", isinstance(get_booking_local, AgentTool))

print()
print("Gateway-discovered tool class:")
print("  chain:", " -> ".join(inheritance_chain(MCPAgentTool)))
print("  is AgentTool:", issubclass(MCPAgentTool, AgentTool))

print()
print("Both satisfy the interface Agent(tools=...) consumes.")
print("Different execution location; the same contract from the agent's perspective.")

### 5c. Put the connection, discovery and agent call in the right lifetime

The exact production flow belongs in one `with` block:

1. Construct `MCPClient` with a transport factory.
2. Enter the context — open the HTTP transport and initialise the MCP session.
3. Call `list_tools_sync()` — ask the Gateway's MCP server for every registered tool and adapt each to
   `MCPAgentTool`.
4. Build the Strands agent with that list, then invoke it **while the client remains open**.
5. Leave the context — close the session and background thread, even if an exception occurred.

That fourth point is load-bearing. Do not return `tools` from the context and invoke them later: an
`MCPAgentTool` calls back through its `MCPClient`, and that client has already been stopped after `__exit__`.
Discovery and use share one session lifetime.

The function below is the demo's entrypoint logic in a reusable shape. It is defined but not called because the
Gateway endpoint does not exist yet. Notice there is no `@tool` anywhere.

> **Note:** `mcp` 1.29.1 exports both `streamable_http_client` and `streamablehttp_client`, and they are
> different functions with different parameter surfaces — not spelling aliases. This course's `demo.py` uses
> `streamable_http_client`, so the notebook uses that exact function rather than substituting the similar name.

Free — definition only; no client is entered.

In [ ]:
# =====================================================
# 5c. THE COMPLETE AGENT-SIDE FLOW, DEFINED BUT NOT CALLED
# =====================================================
from strands import Agent


GATEWAY_SYSTEM_PROMPT = """You are WanderBot, the AI travel assistant for Horizon Travel.

Use the Gateway tools to retrieve bookings by reference or customer email.
Present results clearly, and never invent booking details."""


def invoke_through_gateway(endpoint, user_message, model):
    """Discover Gateway tools and answer one user message while connected.

    The client, discovered tools and agent invocation all remain inside one
    context-managed MCP session. Leaving the block closes the transport and its
    background thread automatically.

    Args:
        endpoint: The Gateway MCP URL, ending in /mcp.
        user_message: The customer's question.
        model: A configured Strands model provider, such as BedrockModel.

    Returns:
        AgentResult: The final response from the Strands agent.
    """
    gateway_client = MCPClient(
        lambda: streamable_http_client(url=endpoint)
    )

    with gateway_client:
        tools = gateway_client.list_tools_sync()
        print("discovered tools:", [tool.tool_name for tool in tools])

        agent = Agent(
            model=model,
            system_prompt=GATEWAY_SYSTEM_PROMPT,
            tools=tools,
        )

        # Invoke before leaving the block. Remote tools need this open client.
        response = agent(user_message)

    return response


print("invoke_through_gateway defined, not called.")
print("No endpoint was opened and no tools were discovered yet.")

### What the output proved

**Construction is not connection.** `MCPClient(make_transport)` left the factory count at zero. Calling
`list_tools_sync()` before `with client:` raised `MCPClientInitializationError: the client session is not
running`, and the count stayed zero afterwards — the refusal was local, with no request to the placeholder.

**The context manager is the session boundary.** From the installed source, `__enter__` calls `start()`, which
starts a background thread and waits for MCP initialisation; `__exit__` calls `stop()`, which closes the session,
event loop and thread. That is why discovery *and tool use* belong inside the same block.

**Local and remote tools meet at the same interface.** The local `DecoratedFunctionTool` and remote
`MCPAgentTool` both inherit `AgentTool`. So this line needs no special Gateway mode:

``` Python
agent = Agent(tools=tools)
```

The model receives names, descriptions and JSON schemas in either case. It does not know whether executing one
runs a local function, crosses an MCP connection to a Gateway, or ends in a Lambda. That location is an
implementation detail below the `AgentTool` boundary.

What Part A cannot prove is equally important: there is no server yet, so `list_tools_sync()` has not returned
a real tool, no MCP handshake has completed, and no Gateway has invoked the Lambda. Those are Part B.

# Part A checkpoint — what is proven before creating anything

| Step | Question | Answer established locally |
| --- | --- | --- |
| 1 | will the notebook run safely? | credentials resolve to the lab account; outputs mask identifiers |
| 2 | why move tools out? | local tools couple logic, dependencies, permissions and deployment to the agent |
| 3 | what is behind the Gateway? | ordinary Lambda functions plus a handler that routes the Gateway's tool name |
| 4 | what does the schema contribute? | the same name/description/input-schema contract `@tool` generated locally |
| 5 | how does the agent consume remote tools? | an MCP session discovers `MCPAgentTool` objects implementing `AgentTool` |

The architecture is now explained from both ends without deploying a Lambda or creating a Gateway:

```
Agent(tools=[MCPAgentTool, ...])
    |  MCP over streamable HTTP  - owned by MCPClient's with block
AgentCore Gateway
    |  validates arguments from booking_lambda.json
    |  sets bedrockAgentCoreToolName in Lambda client context
Lambda handler
    |  strips target___ prefix and routes by tool name
get_booking / list_bookings_by_email
```

## Part B gate

Part B creates real resources and proves the missing links: deploy the Lambda, create a Gateway with a Lambda
target and this schema, copy its `/mcp` endpoint, run `list_tools_sync()`, and make a real booking lookup.

The course uses the AgentCore console and **No Authentication** for development. An unauthenticated Gateway URL
is a capability: anyone holding it can invoke the exposed tools, so it must not be hard-coded into this public
notebook or saved in output. Part B will request it interactively for the kernel's lifetime and will mask it
when printed.

Stop here until those resources are deliberately created.